# Silver: Sold Objects

**Layer:** Silver | **Source:** `raw/sold/all/*.csv` | **Target:** `silver.Fact_SoldObjects`  
**Owner:** christopher.furu@hotmail.com

## Pipeline Steps
1. Load raw CSVs from ADLS → `raw_loaded` temp view
2. Select, rename, cast, normalize → `transformed` temp view (SparkSQL)
3. Filter invalid rows → `filtered` temp view (SparkSQL)
4. Deduplicate → `deduplicated` temp view (SparkSQL)
5. MERGE into Delta table (SparkSQL)
6. Cleanup raw files

## Parameters
| Widget | Default | Description |
|--------|---------|-------------|
| `raw_container` | `raw` | ADLS container for raw data |
| `silver_container` | `silver` | ADLS container for silver tables |

In [ ]:
import sys
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, FloatType
from pyspark.sql.functions import input_file_name
from functools import reduce

_utils_dir = "/Workspace/Repos/christopher.furu@hotmail.com/DataEngineering/databricks/real_estate/notebooks/utils"
if _utils_dir not in sys.path:
    sys.path.insert(0, _utils_dir)

import pipeline_helpers
from pipeline_helpers import log_step

In [ ]:
%run ../utils/pyutils

In [ ]:
%run ../utils/udf_helpers

# Configuration

In [ ]:
dbutils.widgets.text("raw_container", "raw", "Raw container name")
dbutils.widgets.text("silver_container", "silver", "Silver container name")

raw_container = dbutils.widgets.get("raw_container")
silver_container = dbutils.widgets.get("silver_container")
ds_raw = get_wasbs_path(container=raw_container)
ds_silver = get_wasbs_path(container=silver_container)

TYPE_MAP = {
    "IntegerType": IntegerType,
    "StringType": StringType,
    "DoubleType": DoubleType,
    "FloatType": FloatType,
}
sold_schema = StructType([
    StructField(name, TYPE_MAP[type_name](), True)
    for name, type_name in pipeline_helpers.SOLD_SCHEMA_FIELDS
])

log_step("CONFIG", f"raw={ds_raw}  silver={ds_silver}")

# Step 1: Load Raw Data

In [ ]:
files = [f.path for f in dbutils.fs.ls(f"{ds_raw}/sold/all/") if f.path.endswith(".csv")]

if not files:
    log_step("LOAD", "No raw CSV files found. Exiting notebook.")
    dbutils.notebook.exit("NO_DATA")

dfs = [
    spark.read
         .format("csv")
         .option("header", True)
         .option("inferSchema", False)
         .schema(sold_schema)
         .load(path)
         .withColumn("source_file", input_file_name())
    for path in files
]
df_raw = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), dfs)
df_raw.createOrReplaceTempView("raw_loaded")

log_step("LOAD", f"Loaded {df_raw.count()} rows from {len(files)} files into raw_loaded")

# Step 2: Select, Rename, Cast, Normalize

In [ ]:
transform_sql = pipeline_helpers.build_select_rename_cast_sql(
    source_view="raw_loaded",
    include_source_file=True,
)
print(transform_sql)

In [ ]:
spark.sql(transform_sql).createOrReplaceTempView("transformed")
log_step("TRANSFORM", f"{spark.sql('SELECT COUNT(*) AS cnt FROM transformed').collect()[0]['cnt']} rows in transformed")

# Step 3: Filter

In [ ]:
filter_sql = pipeline_helpers.build_filter_sql(
    source_view="transformed",
    extra_filters=['LOWER(url) NOT LIKE "%annons%"'],
)
print(filter_sql)

In [ ]:
spark.sql(filter_sql).createOrReplaceTempView("filtered")
log_step("FILTER", f"{spark.sql('SELECT COUNT(*) AS cnt FROM filtered').collect()[0]['cnt']} rows after filtering")

# Step 4: Deduplicate

In [ ]:
spark.sql("""
    SELECT *
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY booliId, soldDate
                ORDER BY soldPrice DESC
            ) AS row_rank
        FROM filtered
    )
    WHERE row_rank = 1
""").drop("row_rank").createOrReplaceTempView("deduplicated")

log_step("DEDUP", f"{spark.sql('SELECT COUNT(*) AS cnt FROM deduplicated').collect()[0]['cnt']} rows after deduplication")

# Step 5: Merge into Silver

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

create_table_query = f"""
CREATE TABLE IF NOT EXISTS silver.Fact_SoldObjects
(
  booliId INT,
  constructionYear INT,
  daysActive INT,
  soldDate TIMESTAMP,
  latitude FLOAT,
  longitude FLOAT,
  url STRING,
  typeName STRING,
  rent INT,
  floor FLOAT,
  soldSqmPrice FLOAT,
  livingArea FLOAT,
  rooms FLOAT,
  listPrice INT,
  soldPrice INT,
  sourceFileName STRING
)
USING DELTA
LOCATION '{ds_silver}/soldObjects'
"""
spark.sql(create_table_query)
log_step("TABLE", "silver.Fact_SoldObjects ensured")

In [ ]:
try:
    spark.sql("""
        MERGE INTO silver.Fact_SoldObjects AS T
        USING deduplicated AS S
        ON T.booliId = S.booliId AND T.soldDate = S.soldDate
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    log_step("MERGE", "Completed successfully")
except Exception as e:
    log_step("MERGE_FAILED", str(e))
    raise

# Step 6: Validate

In [ ]:
row_count = spark.sql("SELECT COUNT(*) AS cnt FROM silver.Fact_SoldObjects").collect()[0]["cnt"]
log_step("VALIDATE", f"silver.Fact_SoldObjects total rows: {row_count}")
spark.sql("DESCRIBE HISTORY silver.Fact_SoldObjects").show(5, truncate=False)

# Step 7: Cleanup Raw Files

In [ ]:
try:
    if len(files) > 0:
        dbutils.fs.rm(f"{ds_raw}/sold/", recurse=True)
        log_step("CLEANUP", f"Removed raw files from {ds_raw}/sold/")
    else:
        log_step("CLEANUP", "Skipped: no raw files processed")
except Exception as e:
    log_step("CLEANUP_FAILED", str(e))
    # Don't re-raise: data is already merged, cleanup can be retried manually